# 先导 r1 的评测图

先导（2026-10-17 夜）四臂各跑一次的结果图，用来在正式实验之前试图的形式与读法。正式实验的图读 `pweval.measure` 产出的表；先导没有走 `eval/` 的运行框架，表由 `tmps/pilot/tables.py` 产出。

- 数据：`~/pw-lab/pilot/r1/tables/`（仓库外）。重新生成（仓库根目录）：

  ```
  uv run --frozen python tmps/pilot/audit.py r1 a3 a2 a0
  uv run --frozen python tmps/pilot/tables.py r1
  ```

- 本 notebook 只读表、画图，不算任何报告中引用的数字；图只留在 notebook 里，不写文件。
- 读图时的前提（详见 `tmps/pilot/report.md`、`decisions.md`）：
  - 每臂 $n=1$，单题、单个写者的差别噪声很大。
  - 四臂同时跑、共用一台 sglang，秒数互相干扰；token 与调用数更可靠。
  - A4 的写者代价分两部分：pi 会话，加 Graphiti 服务端抽取用的 LLM（sglang :30001，按时间窗分摊到写者，见 `tables.py`）。读者的搜索不调 LLM。嵌入（:8002）各臂都没算。
  - A3、A2 的 FiLM v4 试了 3 次（中间修了 bug、收窄了任务文字），A0、A4 只试了 1 次。
  - A4 第二组有读者按事实里的绝对路径读了论文原文（q8、q11、q12）。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap, LogNorm

TABLES = Path.home() / "pw-lab/pilot/r1/tables"
sessions = pd.read_csv(TABLES / "sessions.csv")
grades = pd.read_csv(TABLES / "grades.csv")
health = pd.read_csv(TABLES / "health.csv")

ARMS = ["a3", "a2", "a0", "a4"]
LABEL = {"a3": "A3 PaperWeave", "a2": "A2 mechanisms off", "a0": "A0 files + Git", "a4": "A4 Graphiti"}
SHORT = {"a3": "A3\nPaperWeave", "a2": "A2\nno mech.", "a0": "A0\nfiles + Git", "a4": "A4\nGraphiti"}
COLOR = {"a3": "#2b6cb0", "a2": "#90b4dc", "a0": "#c0392b", "a4": "#e69138"}
TASK = {
    "build-informer": "Informer", "build-autoformer": "Autoformer", "build-patchtst": "PatchTST",
    "build-tide": "TiDE", "leaderboard": "Leaderboard", "build-timemachine": "TimeMachine",
    "compare": "Compare", "verify-own": "Verify", "verify-later": "Verify later",
    "build-unitime": "UniTime", "build-cats": "CATS", "build-film": "FiLM v3",
    "leaderboard-update": "LB update", "film-v4": "FiLM v4",
}

plt.rcParams.update({
    "figure.dpi": 120, "font.size": 9, "axes.titlesize": 10, "axes.spines.top": False,
    "axes.spines.right": False, "legend.frameon": False,
})

sessions[["server_input", "server_output"]] = sessions[["server_input", "server_output"]].fillna(0)
sessions["tokens_in"] = sessions["input"] + sessions["cache_read"]
sessions["tokens_in_all"] = sessions["tokens_in"] + sessions["server_input"]  # A4 含服务端抽取，其他臂与 tokens_in 相同
sessions["output_all"] = sessions["output"] + sessions["server_output"]
writers = sessions[sessions.role == "writer"].copy()
writers["no"] = writers.who.str.split("-").str[1].astype(int)
readers = sessions[(sessions.role == "reader") & (sessions.variant == "final")].copy()

## 1 读者正确率

每题 0 / 0.5 / 1，第一组 7 题、第二组 5 题。左图只算每臂第一次 v4 尝试之后的库（四臂条件相同），右图算最后一次。斜线是 A4 读者越界读原文得到的分。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.5, 3), sharey=True)
for ax, col, title in zip(axes, ["score_first", "score_final"], ["First v4 attempt (equal terms)", "All attempts"]):
    for i, arm in enumerate(ARMS):
        g = grades[grades.arm == arm]
        w1 = g[g.workload == "w1"][col].sum()
        w2_clean = g[(g.workload == "w2") & (g.leak == 0)][col].sum()
        w2_leak = g[(g.workload == "w2") & (g.leak == 1)][col].sum()
        ax.bar(i, w1, color=COLOR[arm])
        ax.bar(i, w2_clean, bottom=w1, color=COLOR[arm], alpha=0.45)
        if w2_leak:
            ax.bar(i, w2_leak, bottom=w1 + w2_clean, color=COLOR[arm], alpha=0.45, hatch="///", edgecolor="white")
        ax.text(i, w1 + w2_clean + w2_leak + 0.2, f"{w1 + w2_clean + w2_leak:g}", ha="center")
    ax.axhline(7, color="grey", lw=0.6, ls=":")
    ax.set_xticks(range(len(ARMS)), [SHORT[a] for a in ARMS])
    ax.set_ylim(0, 12.8)
    ax.set_title(title)
axes[0].set_ylabel("Probe score (max 12)")
axes[1].legend([plt.Rectangle((0, 0), 1, 1, color="dimgray"), plt.Rectangle((0, 0), 1, 1, color="dimgray", alpha=0.45),
                plt.Rectangle((0, 0), 1, 1, facecolor="dimgray", alpha=0.45, hatch="///", edgecolor="white")],
               ["w1 (7 probes)", "w2 (5 probes)", "w2, reader read the paper"], loc="lower right", fontsize=7)
fig.tight_layout()

## 2 写者有没有把工作写进库

每格一位写者（一个新会话），数字是写库次数（A3、A2 是提交数，A0 是 git 提交数，A4 是 add / delete 调用数）。w2 的失败几乎都来自 pi 自动压缩失败后会话结束（O14）。

In [ ]:
OUTCOME = ["wrote", "edited, not committed", "timeout (2400 s), nothing written", "ended early, nothing written"]
OUT_COLOR = ["#5aa469", "#f0b429", "#8b1e1e", "#d98880"]


def outcome(r):
    if r.writes > 0:
        return 0
    if r.state == "timeout":
        return 2
    return 1 if r.uncommitted else 3


cols = sorted(writers.no.unique())
grid = np.full((len(ARMS), len(cols)), np.nan)
text = {}
for _, r in writers.iterrows():
    i, j = ARMS.index(r.arm), cols.index(r.no)
    grid[i, j] = outcome(r)
    text[i, j] = str(r.writes)

fig, ax = plt.subplots(figsize=(8.5, 2.4))
ax.imshow(np.ma.masked_invalid(grid), cmap=ListedColormap(OUT_COLOR), vmin=-0.5, vmax=3.5, aspect="auto")
for (i, j), t in text.items():
    ax.text(j, i, t, ha="center", va="center", fontsize=7, color="white" if grid[i, j] in (0, 2) else "black")
task_of = writers.drop_duplicates("no").set_index("no").variant.map(TASK)
ax.set_xticks(range(len(cols)), [f"{n} {task_of[n]}" + (f" #{n - 13}" if n >= 14 else "") for n in cols],
              rotation=60, ha="right", fontsize=7)
ax.set_yticks(range(len(ARMS)), [LABEL[a] for a in ARMS])
for x in (8.5, 12.5):
    ax.axvline(x, color="white", lw=3)
ax.spines[:].set_visible(False)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in OUT_COLOR]
ax.legend(handles, OUTCOME, ncol=4, loc="lower center", bbox_to_anchor=(0.5, 1.02), fontsize=7)
fig.tight_layout()

## 3 写者代价：逐个写者与 A3 比

第一组 9 位写者，每点一位写者，纵轴是该写者在某臂的 token 与 A3 同一任务的比值（对数轴），横线是中位数。看差距是不是普遍存在，而不是被个别写者拉出来的。A4 分两列：只算 pi 会话，与加上服务端抽取（工具调用不含服务端）。A4 的写者 9 没写进库，服务端为 0。

In [ ]:
w1 = writers[writers.workload == "w1"].pivot_table(
    index="no", columns="arm", values=["tokens_in", "tokens_in_all", "output", "output_all", "calls"])
COLS = [("a2", "", "A2"), ("a0", "", "A0"), ("a4", "", "A4\npi"), ("a4", "_all", "A4\n+server")]
fig, axes = plt.subplots(1, 3, figsize=(9, 2.9), sharey=True)
for ax, metric, title in zip(axes, ["tokens_in", "output", "calls"],
                             ["Input tokens (incl. cache)", "Output tokens", "Tool calls"]):
    cols = COLS if metric != "calls" else COLS[:3]
    for k, (arm, suffix, _) in enumerate(cols):
        ratio = w1[metric + suffix][arm] / w1[metric]["a3"]
        x = k + np.linspace(-0.18, 0.18, len(ratio))
        ax.scatter(x, ratio, s=14, color=COLOR[arm], zorder=3, alpha=1 if suffix else 0.6 if arm == "a4" else 1,
                   facecolors="none" if arm == "a4" and not suffix and metric != "calls" else COLOR[arm])
        ax.hlines(ratio.median(), k - 0.3, k + 0.3, color="black", lw=1.2)
        ax.text(k + 0.32, ratio.median(), f"{ratio.median():.2g}x", va="center", fontsize=7)
    ax.axhline(1, color="grey", lw=0.6, ls=":")
    ax.set_yscale("log", base=2)
    ax.set_yticks([0.5, 1, 2, 4], ["0.5", "1", "2", "4"])
    ax.set_xticks(range(len(cols)), [c[2] for c in cols])
    ax.set_title(title)
axes[0].set_ylabel("Arm / A3, same writer task")
fig.tight_layout()

## 4 写者代价随写者累积

按写者顺序累加输入 token（含缓存读）与输出 token。圆点是写进了库的会话，叉是没写进的。A4 的虚线加上了服务端抽取。虚线分开 w1、w2 与 FiLM v4。正式实验要画的是同一条曲线在更大库上的斜率。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.5, 3))
for ax, metric, ylabel in zip(axes, ["tokens_in", "output"], ["Cumulative input tokens (M)", "Cumulative output tokens (M)"]):
    for arm in ARMS:
        w = writers[writers.arm == arm].sort_values("no")
        y = w[metric].cumsum() / 1e6
        ax.plot(w.no, y, color=COLOR[arm], lw=1.3, label=LABEL[arm])
        ok = w.writes > 0
        ax.scatter(w.no[ok], y[ok], s=12, color=COLOR[arm], zorder=3)
        ax.scatter(w.no[~ok], y[~ok], s=28, marker="x", color=COLOR[arm], zorder=3)
        if arm == "a4":
            y_all = w[metric.replace("tokens_in", "tokens_in_all") if metric == "tokens_in" else "output_all"].cumsum() / 1e6
            ax.plot(w.no, y_all, color=COLOR[arm], lw=1.3, ls="--", label="A4 + server extraction")
    for x in (9.5, 13.5):
        ax.axvline(x, color="grey", lw=0.6, ls="--")
    ax.set_xlabel("Writer")
    ax.set_ylabel(ylabel)
    ax.set_xticks(range(1, 17))
axes[0].legend(fontsize=7)
fig.tight_layout()

cost = (writers.groupby(["arm", "workload"])[["tokens_in", "server_input", "output", "server_output", "calls"]].sum()
        .div([1e3, 1e3, 1e3, 1e3, 1]).round().astype(int).unstack("workload").reindex(ARMS))
cost  # 千 token；calls 是 pi 的工具调用

## 5 读者代价：逐题

每格是一题的读者输入 token（千，含缓存读；颜色是对数刻度），下面一行是得分。A3、A2 的 q11、q12 是最后一次 v4 尝试之后的库。

In [ ]:
probes = [f"q{i}" for i in range(1, 13)]
tok = readers.pivot_table(index="arm", columns="who", values="tokens_in").reindex(index=ARMS, columns=probes) / 1e3
score = grades.pivot_table(index="arm", columns="probe", values="score_final").reindex(index=ARMS, columns=probes)
state = readers.pivot_table(index="arm", columns="who", values="state", aggfunc="first").reindex(index=ARMS, columns=probes)

fig, ax = plt.subplots(figsize=(8.5, 2.6))
im = ax.imshow(tok.values, cmap="Blues", norm=LogNorm(vmin=30, vmax=2500), aspect="auto")
for i in range(len(ARMS)):
    for j in range(len(probes)):
        v = tok.values[i, j]
        mark = " t/o" if state.values[i, j] == "timeout" else ""
        ax.text(j, i, f"{v:.0f}k\n{score.values[i, j]:g}{mark}", ha="center", va="center", fontsize=6.5,
                color="white" if v > 600 else "black")
ax.axvline(6.5, color="white", lw=3)
ax.set_xticks(range(len(probes)), probes)
ax.set_yticks(range(len(ARMS)), [LABEL[a] for a in ARMS])
ax.spines[:].set_visible(False)
fig.colorbar(im, ax=ax, label="k tokens", fraction=0.03)
fig.tight_layout()

summary = (readers.groupby(["arm", "workload"])[["tokens_in", "calls", "seconds"]].sum()
           .assign(tokens_in=lambda d: (d.tokens_in / 1e3).round()).unstack("workload").reindex(ARMS))
summary

## 6 库的健康（w2 末）

A3、A2、A0 由 `audit.py` 读最终的库（A0 只读已提交的状态）；A4 没有审计，只有人工核对的几项。重复只在手写的规范名表（约 25 个方法、9 个数据集）内查，误合并只查 Weather 一例，所以 0 只表示“表内没查到”。

In [ ]:
METRICS = ["papers_built", "leaderboard_papers", "duplicate_extra_nodes", "wrong_merges_checked",
           "dangling_refs", "unparsable_files", "etth1_method_recall", "commits", "nodes"]
table = health.pivot_table(index="metric", columns="arm", values="value", aggfunc="first").reindex(index=METRICS, columns=ARMS)
table.columns = [LABEL[a] for a in ARMS]
manual = health[health.source != "audit.py"][["arm", "metric", "source"]]
display(table.style.format(lambda v: "" if pd.isna(v) else f"{v:g}" if v != int(v) else f"{int(v)}", na_rep="—"))
manual